In [76]:
import cv2
import numpy as np

from onnxruntime import InferenceSession

In [77]:
session = InferenceSession("../model/best.onnx", providers=["CPUExecutionProvider"])
session

In [78]:
input_name = session.get_inputs()[0].name
input_name

'images'

In [79]:
# Prepare image
img = cv2.imread('987_1_V2_45_NI.jpg')

h, w = img.shape[:2]
print(w, h)

img_resized = cv2.resize(img, (640, 640))
img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
img_normalized = img_rgb.astype(np.float32) / 255.0
img_transposed = np.transpose(img_normalized, (2, 0, 1))
img_input = np.expand_dims(img_transposed, axis=0)

6144 8192


In [80]:
img_input.shape

(1, 3, 640, 640)

In [81]:
# Inference
outputs = session.run(None, {input_name: img_input})

In [82]:
print(outputs)
print(outputs[0].shape)

[array([[[8.50551033e+00, 1.73779259e+01, 2.12677765e+01, ...,
         5.06820923e+02, 5.33767761e+02, 5.66096802e+02],
        [4.30540085e+00, 3.15564489e+00, 2.66883135e+00, ...,
         6.24612366e+02, 6.20265625e+02, 6.13575195e+02],
        [2.01848106e+01, 3.60573730e+01, 4.74386749e+01, ...,
         1.61458710e+02, 1.65031616e+02, 1.90246277e+02],
        [9.20464706e+00, 6.28600311e+00, 5.54395580e+00, ...,
         1.33868652e+02, 1.24907166e+02, 1.27140686e+02],
        [1.42094493e-03, 8.19623470e-04, 4.40567732e-04, ...,
         1.30152702e-03, 1.34670734e-03, 1.46374106e-03]]], dtype=float32)]
(1, 5, 8400)


In [83]:
results = outputs[0][0]
results = results.transpose()

results.shape

(8400, 5)

In [84]:
results

array([[8.50551033e+00, 4.30540085e+00, 2.01848106e+01, 9.20464706e+00,
        1.42094493e-03],
       [1.73779259e+01, 3.15564489e+00, 3.60573730e+01, 6.28600311e+00,
        8.19623470e-04],
       [2.12677765e+01, 2.66883135e+00, 4.74386749e+01, 5.54395580e+00,
        4.40567732e-04],
       ...,
       [5.06820923e+02, 6.24612366e+02, 1.61458710e+02, 1.33868652e+02,
        1.30152702e-03],
       [5.33767761e+02, 6.20265625e+02, 1.65031616e+02, 1.24907166e+02,
        1.34670734e-03],
       [5.66096802e+02, 6.13575195e+02, 1.90246277e+02, 1.27140686e+02,
        1.46374106e-03]], dtype=float32)

In [85]:
def filter_Detections(results, thresh = 0.5):
    
    # if model is trained on 1 class only
    if len(results[0]) == 5:
        # filter out the detections with confidence > thresh
        considerable_detections = [detection for detection in results if detection[4] > thresh]
        considerable_detections = np.array(considerable_detections)
        return considerable_detections

    # if model is trained on multiple classes
    else:
        A = []
        for detection in results:

            class_id = detection[4:].argmax()
            confidence_score = detection[4:].max()

            new_detection = np.append(detection[:4],[class_id,confidence_score])

            A.append(new_detection)

        A = np.array(A)

        # filter out the detections with confidence > thresh
        considerable_detections = [detection for detection in A if detection[-1] > thresh]
        considerable_detections = np.array(considerable_detections)

        return considerable_detections

In [86]:
filter_Detections(results)

array([[361.96774   , 449.26202   ,  43.599915  ,  33.65811   ,
          0.80461013],
       [361.97498   , 449.182     ,  43.872803  ,  33.99698   ,
          0.9154602 ],
       [361.8476    , 449.31677   ,  43.714417  ,  34.126343  ,
          0.9507394 ],
       [361.85175   , 449.31107   ,  43.598663  ,  34.088074  ,
          0.91355395],
       [361.82965   , 449.4793    ,  43.49585   ,  33.83722   ,
          0.90727174],
       [361.9124    , 449.40076   ,  43.947205  ,  33.832306  ,
          0.976616  ],
       [361.85767   , 449.28546   ,  43.570923  ,  33.687866  ,
          0.90888166],
       [361.88745   , 449.3294    ,  43.563995  ,  33.848846  ,
          0.93990505],
       [361.836     , 448.92026   ,  43.71115   ,  34.268494  ,
          0.95980287],
       [361.86765   , 448.992     ,  43.59192   ,  34.166046  ,
          0.90543586]], dtype=float32)

In [87]:
def NMS(boxes, conf_scores, iou_thresh = 0.55):

    #  boxes [[x1,y1, x2,y2], [x1,y1, x2,y2], ...]

    x1 = boxes[:,0]
    y1 = boxes[:,1]
    x2 = boxes[:,2]
    y2 = boxes[:,3]

    areas = (x2-x1)*(y2-y1)

    order = conf_scores.argsort()

    keep = []
    keep_confidences = []

    while len(order) > 0:
        idx = order[-1]
        A = boxes[idx]
        conf = float(conf_scores[idx])

        order = order[:-1]

        xx1 = np.take(x1, indices= order)
        yy1 = np.take(y1, indices= order)
        xx2 = np.take(x2, indices= order)
        yy2 = np.take(y2, indices= order)

        keep.append(A)
        keep_confidences.append(conf)

        # iou = inter/union

        xx1 = np.maximum(x1[idx], xx1)
        yy1 = np.maximum(y1[idx], yy1)
        xx2 = np.minimum(x2[idx], xx2)
        yy2 = np.minimum(y2[idx], yy2)

        w = np.maximum(xx2-xx1, 0)
        h = np.maximum(yy2-yy1, 0)

        intersection = w*h

        # union = areaA + other_areas - intesection
        other_areas = np.take(areas, indices= order)
        union = areas[idx] + other_areas - intersection

        iou = intersection/union

        boleans = iou < iou_thresh

        order = order[boleans]

        # order = [2,0,1]  boleans = [True, False, True]
        # order = [2,1]

    return keep, keep_confidences

# Hutang: try to remember how NMS works !

In [88]:
detected_box = filter_Detections(results)
NMS(detected_box[:,0:4], detected_box[:,4])

([array([361.9124  , 449.40076 ,  43.947205,  33.832306], dtype=float32),
  array([361.836   , 448.92026 ,  43.71115 ,  34.268494], dtype=float32),
  array([361.8476  , 449.31677 ,  43.714417,  34.126343], dtype=float32),
  array([361.88745 , 449.3294  ,  43.563995,  33.848846], dtype=float32),
  array([361.97498 , 449.182   ,  43.872803,  33.99698 ], dtype=float32),
  array([361.85175 , 449.31107 ,  43.598663,  34.088074], dtype=float32),
  array([361.85767 , 449.28546 ,  43.570923,  33.687866], dtype=float32),
  array([361.82965, 449.4793 ,  43.49585,  33.83722], dtype=float32),
  array([361.86765 , 448.992   ,  43.59192 ,  34.166046], dtype=float32),
  array([361.96774 , 449.26202 ,  43.599915,  33.65811 ], dtype=float32)],
 [0.9766160249710083,
  0.9598028659820557,
  0.9507393836975098,
  0.939905047416687,
  0.9154602289199829,
  0.9135539531707764,
  0.908881664276123,
  0.9072717428207397,
  0.9054358601570129,
  0.8046101331710815])

In [89]:
# function to rescale bounding boxes 
def rescale_back(results,img_w,img_h):
    cx, cy, w, h, class_id, confidence = results[:,0], results[:,1], results[:,2], results[:,3], results[:,4], results[:,-1]
    cx = cx/640.0 * img_w
    cy = cy/640.0 * img_h
    w = w/640.0 * img_w
    h = h/640.0 * img_h
    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2

    boxes = np.column_stack((x1, y1, x2, y2, class_id))
    keep, keep_confidences = NMS(boxes,confidence)
    print(np.array(keep).shape)
    return keep, keep_confidences

In [90]:
rescaled, conf = rescale_back(detected_box, w, h)

(1, 5)


In [91]:
rescaled

[array([3.2634128e+03, 5.5358027e+03, 3.6853059e+03, 5.9688564e+03,
        9.7661602e-01], dtype=float32)]

In [92]:
conf

[0.9766160249710083]

In [93]:
print(type(rescaled))
print(type(conf))

<class 'list'>
<class 'list'>


In [94]:
[round(c, 2) for c in conf]

[0.98]

In [95]:
[box.tolist() for box in rescaled]

[[3263.412841796875,
  5535.802734375,
  3685.305908203125,
  5968.8564453125,
  0.9766160249710083]]

In [96]:
classes = ["pill"]

In [97]:
# for res, conf in zip(rescaled, conf):

#     x1,y1,x2,y2, cls_id = res
#     cls_id = int(cls_id)
#     x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
#     conf = "{:.2f}".format(conf)
#     # draw the bounding boxes
#     cv2.rectangle(img,(int(x1),int(y1)),(int(x2),int(y2)),(255,0, 0),1)
#     cv2.putText(img,classes[cls_id]+' '+conf,(x1,y1-17),
#                 cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,0,0),1)


# cv2.imwrite("Output.jpg", img)